# Hybrid Quantum Physics‑Informed Neural Network (QPINN)
## 2D Crystal Growth with Phase‑Field Physics (10–16 Qubits)

This notebook implements a **hybrid classical–quantum PINN** for a simplified 2D crystal‑growth model.

Outputs of the neural network:

$$
\mathbf{y}(x,y)=(u,v,p,c,\phi)
$$

where

- \(u,v\): velocity components  
- \(p\): pressure  
- \(c\): concentration  
- \(\phi\): phase field

The phase field distinguishes phases:

- \(\phi \approx 1\) → solid  
- \(\phi \approx -1\) → liquid  
- \(\phi \approx 0\) → interface

## Phase‑field chemical potential

\[
\mu = -\epsilon(\theta)^2 \nabla^2 \phi + \phi(\phi^2-1) -2\lambda_c c \phi
\]

Anisotropic surface energy:

\[
\epsilon(\theta) = \epsilon_0(1+\delta \cos(m\theta))
\]

Angle definition:

\[
\theta = \text{atan2}(\phi_y,\phi_x)
\]

Stefan‑type residual:

\[
R_S = \mu - \lambda_T c |\nabla \phi|
\]

Physics‑informed loss:

\[
L = \langle \mu^2 + R_S^2 + |\nabla \phi|^2 \rangle
\]

In [ ]:
# !pip install torch qiskit qiskit-ibm-runtime
import torch
import torch.nn as nn
import numpy as np
import math

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

In [ ]:
# Configuration

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_QUBITS = 12
N_Q_LAYERS = 3
MAX_Q_FEATURES = 4

N_BULK = 32
N_INTERFACE = 64

EPS0 = 0.01
DELTA_ANISO = 0.05
ANISO_M = 4
LAMBDA_C = 1.0
LAMBDA_T = 1.0

## Classical backbone

A small neural network encodes the spatial coordinates before feeding
them into the quantum circuit.

In [ ]:
class ClassicalBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2,64),
            nn.Tanh(),
            nn.Linear(64,32),
            nn.Tanh()
        )

    def forward(self,x):
        return self.net(x)

## Quantum ansatz

Data re‑uploading variational circuit scalable to **10‑16 qubits**.

Each layer applies

1. feature encoding rotations
2. trainable rotations
3. ring entanglement

In [ ]:
def build_quantum_ansatz(n_qubits,n_layers,feature_dim=4):

    qc = QuantumCircuit(n_qubits)

    input_params=[Parameter(f"x{i}") for i in range(feature_dim)]
    weight_params=[]

    for layer in range(n_layers):

        for q in range(n_qubits):
            x=input_params[q%feature_dim]
            qc.ry(x,q)
            qc.rz(0.5*x,q)

        for q in range(n_qubits):
            for gate in ["rx","ry","rz"]:
                p=Parameter(f"theta_{layer}_{q}_{gate}")
                getattr(qc,gate)(p,q)
                weight_params.append(p)

        for q in range(n_qubits-1):
            qc.cx(q,q+1)
        qc.cx(n_qubits-1,0)

    return qc,input_params,weight_params

## Observables

Instead of using a global observable

$$Z^{\otimes n} $$

we measure **local expectation values**

$$
(\langle Z_1\rangle,...,\langle Z_n\rangle)
$$
Afra Wohlschläger
A Kuramoto-Approach to EEG Analysis

In [ ]:
def build_local_observables(n_qubits):

    obs=[]

    for i in range(n_qubits):
        pauli=["I"]*n_qubits
        pauli[n_qubits-1-i]="Z"
        obs.append(SparsePauliOp.from_list([("".join(pauli),1.0)]))

    return obs

In [ ]:
class LocalQuantumLayer(nn.Module):

    def __init__(self,qc,input_params,weight_params,n_qubits):
        super().__init__()

        self.qc=qc
        self.input_params=input_params
        self.weight_params=weight_params
        self.n_qubits=n_qubits

        self.estimator=StatevectorEstimator()
        self.observables=build_local_observables(n_qubits)

        self.weights=nn.Parameter(0.01*torch.randn(len(weight_params)))

    def bind(self,xi):
        values=list(xi.detach().cpu().numpy())
        values+=list(self.weights.detach().cpu().numpy())
        params=self.input_params+self.weight_params
        return {p:v for p,v in zip(params,values)}

    def forward(self,x):

        outputs=[]

        for xi in x:

            bound=self.qc.assign_parameters(self.bind(xi))

            row=[]

            for obs in self.observables:
                job=self.estimator.run([(bound,obs)])
                val=float(job.result()[0].data.evs)
                row.append(val)

            outputs.append(row)

        return torch.tensor(outputs,dtype=torch.float32,device=x.device)

## Hybrid PINN architecture

In [ ]:
class HybridCrystalPINN(nn.Module):

    def __init__(self,qlayer,n_qubits):
        super().__init__()

        self.backbone=ClassicalBackbone()

        self.pre_q=nn.Sequential(
            nn.Linear(32,16),
            nn.Tanh(),
            nn.Linear(16,4)
        )

        self.q=qlayer

        self.post=nn.Sequential(
            nn.Linear(n_qubits,64),
            nn.Tanh(),
            nn.Linear(64,32),
            nn.Tanh(),
            nn.Linear(32,5)
        )

    def forward(self,x):

        z=self.backbone(x)
        q_in=self.pre_q(z)
        q_out=self.q(q_in)

        return self.post(q_out)

## Correct phase‑field derivatives

The gradient must be computed from the **phase field only**:

\[
\phi_x = \frac{\partial \phi}{\partial x}, \quad
\phi_y = \frac{\partial \phi}{\partial y}
\]

Laplacian:

\[
\nabla^2 \phi =
\frac{\partial^2\phi}{\partial x^2} +
\frac{\partial^2\phi}{\partial y^2}
\]

In [ ]:
def anisotropic_epsilon(phi_x,phi_y):
    theta=torch.atan2(phi_y,phi_x+1e-8)
    return EPS0*(1+DELTA_ANISO*torch.cos(ANISO_M*theta))

def phase_field_mu(phi,phi_x,phi_y,lap_phi,c):

    eps=anisotropic_epsilon(phi_x,phi_y)

    return -eps**2*lap_phi + phi*(phi**2-1) -2*LAMBDA_C*c*phi

def stefan_residual(mu,phi_x,phi_y,c):

    grad=torch.sqrt(phi_x**2+phi_y**2+1e-8)
    return mu-LAMBDA_T*c*grad

In [ ]:
def crystal_growth_loss(model,x):

    out=model(x)
    u,v,p,c,phi=out.T

    grad_phi=torch.autograd.grad(
        phi,x,
        torch.ones_like(phi),
        create_graph=True,
        retain_graph=True
    )[0]

    phi_x=grad_phi[:,0]
    phi_y=grad_phi[:,1]

    grad_phi_x=torch.autograd.grad(
        phi_x,x,
        torch.ones_like(phi_x),
        create_graph=True,
        retain_graph=True
    )[0]

    grad_phi_y=torch.autograd.grad(
        phi_y,x,
        torch.ones_like(phi_y),
        create_graph=True,
        retain_graph=True
    )[0]

    lap_phi=grad_phi_x[:,0]+grad_phi_y[:,1]

    mu=phase_field_mu(phi,phi_x,phi_y,lap_phi,c)
    stefan=stefan_residual(mu,phi_x,phi_y,c)

    return (
        mu.pow(2).mean()
        + stefan.pow(2).mean()
        + (phi_x**2+phi_y**2).mean()
    )

In [ ]:
# Build model

qc,x_params,w_params = build_quantum_ansatz(
    N_QUBITS,
    N_Q_LAYERS,
    MAX_Q_FEATURES
)

qlayer = LocalQuantumLayer(
    qc,
    x_params,
    w_params,
    N_QUBITS
)

model = HybridCrystalPINN(
    qlayer,
    N_QUBITS
).to(DEVICE)

print("Qubits:",qc.num_qubits)
print("Quantum parameters:",len(w_params))

In [ ]:
# Quick forward pass test

x=torch.rand(4,2,device=DEVICE,requires_grad=True)

y=model(x)

print("Output shape:",y.shape)